In [5]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('../pc_data.csv')

# Function to safely parse the 'components' column into dictionaries
def safe_parse(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else val
    except (SyntaxError, ValueError):
        return {}

df['components'] = df['components'].apply(safe_parse)

# Convert all component values to strings
df['components'] = df['components'].apply(lambda x: {k: str(v) for k, v in x.items()})

# Convert 'components' into a multi-label format
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(df['components'].apply(lambda x: list(x.values())))

# Convert to DataFrame
y_df = pd.DataFrame(y_encoded, columns=mlb.classes_)

# One-hot encode categorical features
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(df[['use_case', 'category']])

# Create a DataFrame for the encoded features
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out())

# Combine with budget column
X = pd.concat([df[['budget']], encoded_df], axis=1)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y_df, test_size=0.2, random_state=42)

# Train Multi-Label Classification Model (Random Forest)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Compute accuracy (averaging across all labels)
accuracy = (y_pred == y_test).mean().mean()
print(f"Model Accuracy: {accuracy:.4f}")


MemoryError: could not allocate 404619264 bytes